In [1]:
import sys
sys.path.append("../")
# import config
import os
import numpy as np
import random

import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import time

# import DATA_CODE.DATA_LOADER as DATA_LOADER
from torch.utils.data.dataset import Dataset

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define the paths and other parameters
experiment_id = 'train'
sat_data_dir_path = '/content/drive/MyDrive/CalCrop_Data/Satellite/'
label_data_dir_path = '/content/drive/MyDrive/CalCrop_Data/Label_Eroded/'
distri_dir_path = ''
model_dir = '/content/drive/MyDrive/CalCrop_Data/Model'
in_channels = 27
out_channels = 1
learning_rate = 0.0001
n_epochs = 10


input_patch_size = 32
output_patch_size = 32
batch_size = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)



cuda


In [ ]:
class STATT(torch.nn.Module):
    """Spatio-Temporal Attention Network (STATT) for sequence-based image segmentation.

    Architecture:
    - Encoder: Extracts spatial features at multiple scales
    - Temporal Attention: Processes sequence features with LSTM + Attention
    - Decoder: Reconstructs segmentation mask with skip connections
    """

    def __init__(self, in_channels, out_channels, seq_len):
        """Initialize the STATT model.

        Args:
            in_channels: Number of input channels per timestep
            out_channels: Number of output classes
        """
        super(STATT, self).__init__()
        self.seq_len = seq_len

        # --- Encoder Path (Downsampling) ---
        # Block 1: Maintains spatial resolution
        self.conv1_1 = torch.nn.Conv2d(in_channels, 64, 3, padding=1)  # Conv 3x3
        self.conv1_2 = torch.nn.Conv2d(64, 64, 3, padding=1)           # Output: 64 channels

        # Block 2: Doubles features, halves resolution
        self.conv2_1 = torch.nn.Conv2d(64, 128, 3, padding=1)          # Input from pool1
        self.conv2_2 = torch.nn.Conv2d(128, 128, 3, padding=1)         # Output: 128 channels

        # Block 3: Deepest features, halves resolution again
        self.conv3_1 = torch.nn.Conv2d(128, 256, 3, padding=1)         # Input from pool2
        self.conv3_2 = torch.nn.Conv2d(256, 256, 3, padding=1)         # Output: 256 channels

        # --- Decoder Path (Upsampling) ---
        # Upsampling Block 2 (from deepest features)
        self.unpool2 = torch.nn.ConvTranspose2d(512, 128, kernel_size=2, stride=2)  # 2x upsampling
        self.upconv2_1 = torch.nn.Conv2d(256, 128, 3, padding=1)       # After skip connection
        self.upconv2_2 = torch.nn.Conv2d(128, 128, 3, padding=1)       # Output: 128 channels

        # Upsampling Block 1 (final resolution)
        self.unpool1 = torch.nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)  # 2x upsampling
        self.upconv1_1 = torch.nn.Conv2d(128, 64, 3, padding=1)        # After skip connection
        self.upconv1_2 = torch.nn.Conv2d(64, 64, 3, padding=1)         # Output: 64 channels

        # Output layer (classifier)
        self.out = torch.nn.Conv2d(64, out_channels, kernel_size=1, padding=0)  # 1x1 conv

        # --- Shared Operations ---
        self.maxpool = torch.nn.MaxPool2d(2)    # Halves spatial dimensions
        self.relu = torch.nn.ReLU()             # Activation function

        # --- Temporal Processing ---
        self.lstm = torch.nn.LSTM(
            input_size=256,          # Input feature size
            hidden_size=256,          # LSTM hidden state size
            batch_first=True,         # Input shape: (batch, seq, features)
            bidirectional=True       # Concatenates forward/backward outputs
        )  # Output size: 512 (256*2)

        # Attention mechanism
        self.attention = torch.nn.Linear(512, 1)  # Computes attention scores

        # Weight initialization (Xavier uniform)
        for m in self.modules():
            if isinstance(m, torch.nn.Conv2d) or isinstance(m, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)

    def crop_and_concat(self, x1, x2):
        """Aligns and concatenates encoder features (x1) with decoder features (x2).

        Used for skip connections. Center-crops x1 to match x2's spatial dimensions.
        """
        # Calculate cropping offsets for center alignment
        offset_2 = (x1.shape[2] - x2.shape[2]) // 2  # Height offset
        offset_3 = (x1.shape[3] - x2.shape[3]) // 2  # Width offset

        # Crop the larger tensor (x1) to match x2's size
        x1_crop = x1[:, :, offset_2:offset_2+x2.shape[2], offset_3:offset_3+x2.shape[3]]

        # Concatenate along channel dimension
        return torch.cat([x1_crop, x2], dim=1)

    def forward(self, x_s):
        
        batch_size, seq_len, channels, H, W = x_s.shape

        x_s = x_s.reshape(-1, channels, H, W)

           # --- Encoder ---
        conv1 = self.relu(self.conv1_1(x_s))
        conv1 = self.relu(self.conv1_2(conv1))       # [batch*seq_len, 64, H, W]
        pool1 = self.maxpool(conv1)

        conv2 = self.relu(self.conv2_1(pool1))
        conv2 = self.relu(self.conv2_2(conv2))        # [batch*seq_len, 128, H/2, W/2]
        pool2 = self.maxpool(conv2)

        conv3 = self.relu(self.conv3_1(pool2))
        conv3 = self.relu(self.conv3_2(conv3))         # [batch*seq_len, 256, H/4, W/4]

        Hc, Wc = conv3.shape[2], conv3.shape[3]

        # --- Reshape for LSTM: treat every pixel as its own sequence ---
        conv3 = conv3.view(batch_size, seq_len, 256, Hc, Wc)
        conv3 = conv3.permute(0, 3, 4, 1, 2)                    # [batch, Hc, Wc, seq_len, 256]
        conv3 = conv3.reshape(batch_size * Hc * Wc, seq_len, 256)  # [batch*Hc*Wc, seq_len, 256]

        lstm_out, _ = self.lstm(conv3)  # [batch*Hc*Wc, seq_len, 512]

        # --- Attention over the sequence ---
        attn_scores = self.attention(torch.tanh(lstm_out))       # [batch*Hc*Wc, seq_len, 1]
        attn_weights = torch.softmax(attn_scores, dim=1)          # weight per timestep, per pixel

        # Weighted sum across the sequence -> single summary per pixel
        context = torch.sum(attn_weights * lstm_out, dim=1)       # [batch*Hc*Wc, 512]
        context = context.view(batch_size, Hc, Wc, 512).permute(0, 3, 1, 2)  # [batch, 512, Hc, Wc]

        # --- Decoder --- (uses only the LAST frame's skip connections, i.e. the current moment t)
        conv1_last = conv1.view(batch_size, seq_len, 64, H, W)[:, -1]
        conv2_last = conv2.view(batch_size, seq_len, 128, H // 2, W // 2)[:, -1]

        up2 = self.unpool2(context)
        cat2 = self.crop_and_concat(conv2_last, up2)
        up2 = self.relu(self.upconv2_1(cat2))
        up2 = self.relu(self.upconv2_2(up2))

        up1 = self.unpool1(up2)
        cat1 = self.crop_and_concat(conv1_last, up1)
        up1 = self.relu(self.upconv1_1(cat1))
        up1 = self.relu(self.upconv1_2(up1))

        return self.out(up1)  # [batch, out_channels, H, W]

In [ ]:
print("#######################################################################")
print("BUILD MODEL")
# Initialize the STATT model (spatio-temporal attention network)
model = STATT(
    in_channels=in_channels,    # Number of input channels per timestep
    out_channels=out_channels   # Number of output classes for segmentation
)
model = model.to(device)  # Move model to GPU if available



print("#######################################################################")
print("TRAIN MODEL")
# Initialize lists to track training progress
train_loss = []  # Will store average loss per epoch


# Main training loop - iterate through all epochs
for epoch in range(1, n_epochs + 1):
    print('## EPOCH {} ##'.format(epoch))

    # Set model to training mode (enables dropout, updates batch norm stats)
    model.train()
    train_time_start = time.time()  # Start epoch timer

    # Normally you'd have multiple grids, but here using one for demonstration
    dataset = ['T11SKA_2018_1_2']  # Single grid identifier
    dataset = random.sample(dataset, len(dataset))  # Shuffle (trivial with one item)

    epoch_loss = 0  # Accumulate loss across all grids
    # Process each geographic grid
    for grid_num, grid in enumerate(dataset):
        grid_time_start = time.time()  # Start grid timer

        # Create patches for current grid
        image_patches, label_patches = create_patches_satellite_labels(grid)

        # Create PyTorch Dataset
        data = SEGMENTATION(image_patches, label_patches)

        # Create DataLoader for batching
        data_loader = torch.utils.data.DataLoader(
            dataset=data,
            batch_size=batch_size,
            shuffle=True,       # Shuffle patches within grid
            num_workers=0,      # No parallel loading (0 for debugging)
            drop_last=False       # Keep the last incomplete batch
        )

        grid_loss = 0  # Accumulate loss for this grid
        # Process all batches in grid
        for batch, [image_patch, label_patch] in enumerate(data_loader):
            # Reset gradients before each batch
            optimizer.zero_grad()

            # Forward pass - model prediction
            # image_patch shape: [batch, timesteps, channels, H, W]
            out = model(image_patch.to(device))

            # Prepare labels: convert to long tensor and move to device
            label_patch = label_patch.type(torch.long).to(device)

            # Calculate loss between prediction and ground truth
            batch_loss = criterion(out, label_patch)

            # Backpropagation
            batch_loss.backward()       # Calculate gradients
            optimizer.step()            # Update weights

            # Accumulate batch loss
            grid_loss += batch_loss.item()  # Get scalar value from loss tensor

        # Calculate average loss per batch for this grid
        grid_loss = grid_loss / (batch + 1)  # +1 because batch starts at 0
        print('\tGrid No:{}\tGrid:{}\tGrid Loss:{:.4f}\tGrid Time:{:.2f}s'.format(
            grid_num, grid, grid_loss, time.time() - grid_time_start
        ))
        epoch_loss += grid_loss

    # Calculate average loss across all grids
    epoch_loss = epoch_loss / (grid_num + 1)
    print('\nTrain Loss:{:.4f}  Train Time:{:.2f}s'.format(
        epoch_loss, time.time() - train_time_start
    ))
    train_loss.append(epoch_loss)  # Store for later analysis

    if(epoch_loss < train_loss[-1] or len(train_loss)==1): #This is to make sure the model is not overfitting. You usually do this with a validation set and not the train set
      torch.save(model.state_dict(), "Model.pt") #You also might want to consider saving every epochs



#######################################################################
BUILD MODEL
#######################################################################
TRAIN MODEL
## EPOCH 1 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:2.1161	Grid Time:15.75s

Train Loss:2.1161  Train Time:15.75s
## EPOCH 2 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.9806	Grid Time:15.25s

Train Loss:0.9806  Train Time:15.25s
## EPOCH 3 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.8169	Grid Time:13.66s

Train Loss:0.8169  Train Time:13.66s
## EPOCH 4 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7472	Grid Time:13.73s

Train Loss:0.7472  Train Time:13.73s
## EPOCH 5 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7341	Grid Time:13.73s

Train Loss:0.7341  Train Time:13.73s
## EPOCH 6 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7046	Grid Time:13.50s

Train Loss:0.7046  Train Time:13.50s
## EPOCH 7 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.6886	Grid Time:13.85s

Train Loss:0.6886  Train Time:13.85s
## EPO

In [ ]:
print("#######################################################################")
print("TEST MODEL")
model = STATT(in_channels=in_channels, out_channels=out_channels)
model = model.to(device)

print("LOAD MODEL")
model.load_state_dict(torch.load("Model.pt"),strict = False) #This loads the model we saved
# Define loss function (same as training, ignoring unlabeled pixels)
criterion = torch.nn.NSELoss()

print("#######################################################################")

# Initialize metrics storage
test_loss = []    # Track loss per test run
label_list = []   # Collect all ground truth labels
pred_list = []    # Collect all model predictions

# Set model to evaluation mode (disables dropout/BatchNorm)
model.eval()
test_time_start = time.time()  # Start test timer

# Test dataset - normally multiple grids, here using one for demonstration
dataset = ['T11SKA_2019_2_2']  # Grid identifier

epoch_loss = 0  # Accumulate loss across grids
# Process each grid in test dataset
for grid_num, grid in enumerate(dataset):
    # Create patches for current grid (without weather data)
    image_patches, label_patches = create_patches_satellite_labels(grid)

    # Create PyTorch Dataset
    data = SEGMENTATION(image_patches, label_patches)

    # Create DataLoader for batching
    data_loader = torch.utils.data.DataLoader(
        dataset=data,
        batch_size=batch_size,
        shuffle=True,       # Shuffle patches (optional for testing)
        num_workers=0,      # No parallel loading
        drop_last=True       # Ignore last incomplete batch
    )

    grid_loss = 0  # Accumulate loss for this grid
    # Process all batches in grid
    for batch, [image_patch, label_patch] in enumerate(data_loader):
        # Forward pass WITHOUT gradient calculation (saves memory)
        with torch.no_grad():
            patch_out = model(image_patch.to(device))


        # Detach from computation graph and move to CPU
        patch_prob_out_numpy = patch_prob_out.cpu().detach().numpy()

        # Get predicted class (index with highest probability)
        # Shape: [batch, height, width]
        pred_patch = np.argmax(patch_prob_out_numpy, axis=1)

        # Prepare labels for loss calculation
        label_patch_device = label_patch.type(torch.long).to(device)

        # Calculate loss
        batch_loss = criterion(patch_out, label_patch_device)
        grid_loss += batch_loss.item()  # Accumulate batch loss

        # Flatten predictions and labels to 1D arrays
        pred_patch_flat = np.reshape(pred_patch, (-1))        # [batch*height*width]
        label_patch_flat = np.reshape(label_patch, (-1))      # [batch*height*width]

        # Filter out unknown_class pixels (ignore_index)
        valid_mask = label_patch_flat != unknown_class
        pred_grid_flat = pred_patch_flat[valid_mask]
        label_grid_flat = label_patch_flat[valid_mask]

        # Collect valid predictions and labels for overall metrics
        for l in range(pred_grid_flat.shape[0]):
            label_list.append(label_grid_flat[l])
            pred_list.append(pred_grid_flat[l])

    # Calculate average loss for current grid
    grid_loss = grid_loss / (batch + 1)
    print('\tGrid No:{}\tGrid:{}\tGrid Loss:{:.4f}'.format(
        grid_num, grid, grid_loss
    ))
    epoch_loss += grid_loss

# Convert collected results to numpy arrays
label_array = np.array(label_list)  # All ground truth labels
pred_array = np.array(pred_list)    # All model predictions

# Calculate overall test loss
epoch_loss = epoch_loss / (grid_num + 1)
print('Test Loss:{:.4f}  Test Time:{:.2f}s'.format(
    epoch_loss, time.time() - test_time_start
))
test_loss.append(epoch_loss)  # Store for later analysis


# Compute support (i.e., the number of occurrences per class in label_array)
unique_labels, support = np.unique(label_array, return_counts=True)

# Filter labels with support above 50,000
valid_labels = unique_labels[support > threshold]


#######################################################################
TEST MODEL
LOAD MODEL
#######################################################################
	Grid No:0	Grid:T11SKA_2019_2_2	Grid Loss:1.3966
Test Loss:1.3966  Test Time:22.09s


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      Grapes     0.5996    1.0000    0.7497    549852
     Almonds     0.0000    0.0000    0.0000    220570
       Urban     0.0000    0.0000    0.0000     94270

   micro avg     0.5996    0.6359    0.6172    864692
   macro avg     0.1999    0.3333    0.2499    864692
weighted avg     0.3813    0.6359    0.4767    864692



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
